<a href="https://colab.research.google.com/github/MLuc123/ds2002-fa26/blob/main/2026_10_09_public_api_to_clean_table_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [ ]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    delay_seconds = 1

    for i in range(tries):
        r = requests.get(BASE, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return pd.DataFrame(data['daily'])
        elif r.status_code == 400 or r.status_code == 404:
            raise Exception("Client error: " + str(r.status_code))
        elif r.status_code in RETRY_STATUSES:
            time.sleep(delay_seconds)
            delay_seconds = delay_seconds * 2  # Double the wait time
        else:
            raise Exception("Unexpected error: " + str(r.status_code))

    raise Exception("Failed to get weather after " + str(tries) + " attempts.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [ ]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
try:
    raw_wx = get_weather(start='2026-09-05', end='2026-10-03')
    day1_match = raw_wx['time'] == '2026-09-05'
    day2_match = raw_wx['time'] == '2026-10-03'
    wx_fetched = raw_wx[day1_match | day2_match].copy()
except Exception as e:
    print("Error fetching data:", e)
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [ ]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback == True:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = wx_fetched.copy()
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [ ]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [ ]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [ ]:
# TODO
row_0 = wx.iloc[0]
row_1 = wx.iloc[1]
if row_0['precip_mm'] > row_1['precip_mm']:
    wetter_date = row_0['date']
    more_rain = row_0['precip_mm']
    diff = row_0['precip_mm'] - row_1['precip_mm']
else:
    wetter_date = row_1['date']
    more_rain = row_1['precip_mm']
    diff = row_1['precip_mm'] - row_0['precip_mm']
print(str(wetter_date.date()) + " was the wetter date.")
print("It had " + str(more_rain) + " mm of rain, which is " + str(diff) + " mm more than the other date.")

2026-10-03 was the wetter date.
It had 14.7 mm of rain, which is 12.2 mm more than the other date.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [ ]:
# TODO
print("Endpoint: " + BASE)
print("Coordinates: 38.03, -78.51")
print("Requested Date Range: 2026-09-05 to 2026-10-03")

if used_fallback == True:
    print("Data Source: Fallback sample")
else:
    print("Data Source: Real API data")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Requested Date Range: 2026-09-05 to 2026-10-03
Data Source: Real API data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_your answer here_

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [ ]:
# stretch
wx_cville = wx.copy()
wx_cville['city'] = 'Charlottesville'
orlando_used_fallback = False

try:
    raw_orlando = get_weather(start='2026-09-05', end='2026-10-03', lat=28.54, lon=-81.38)
    day1 = raw_orlando['time'] == '2026-09-05'
    day2 = raw_orlando['time'] == '2026-10-03'
    wx_orlando = raw_orlando[day1 | day2].copy()
    wx_orlando['source'] = 'open-meteo'

except Exception as e:
    print("Error fetching Orlando data:", e)
    wx_orlando = FALLBACK.copy()
    wx_orlando['source'] = 'fallback sample'
wx_orlando = wx_orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
wx_orlando['date'] = pd.to_datetime(wx_orlando['date'])
wx_orlando['city'] = 'Orlando'
both_cities = pd.concat([wx_cville, wx_orlando])
both_cities = both_cities[['date', 'city', 'temp_max', 'precip_mm', 'source']]
display(both_cities)

,date,city,temp_max,precip_mm,source
0,2026-09-05,Charlottesville,29.4,2.5,open-meteo
28,2026-10-03,Charlottesville,22.0,14.7,open-meteo
0,2026-09-05,Orlando,32.1,3.9,open-meteo
28,2026-10-03,Orlando,30.4,14.5,open-meteo
